In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# B: fixed saved-JOINT terminal bridge (2E + 4D)

Select a CUDA runtime and Run all. This reuses run `20261009T132316545817Z` JOINT
`terminal_latent.pt` and `float_rgb.pt` from the fixed Drive path. It reads that run only.
Missing/incompatible float input is an engineering failure; no fallback decode is added.
One frozen FP32 Wan VAE is loaded; no transformer/pipeline, sampling, codec or backward runs.
GPU names and reference dependency versions are informational, not admission gates.

Fixed order: read base and original-carrier candidate; encode both; decode
E(candidate), z_T+raw, z_T+mask(raw), z_T+cap(mask(raw)). Key/message/ROIs/rho=.5/cap=1
and known-grid reader stay fixed. All executable layers run despite negative statistics.
The cap and norms describe the terminal intervention, not signal retention.

Each of six layers retains 88 windows × 16 chips (704 state + 704 payload),
energies/support/q, 22 offset correlations, 32 fragment-bit means, paired changes and
weakest bits. Candidate and four decoded layers retain pre/post-clamp observations
from the same calls. Saved base has **no pre-clamp observation**. No OFF/wrong-key grid.

Output is a fresh UTC directory under `MyDrive/Video-WM/Local-Joint-State-Payload-V1-Terminal-Bridge/`.
Return the entire directory, even on failure: setup/fixed slots, source/environment receipts,
execution log/failure, `run/result.json`, per-layer raw/metrics, tensors, paired comparisons.
Counts are saved at model call entry/return; interruption preserves completed observations.
A killed kernel cannot run final cleanup; the durable RUNNING record remains honest until inspected.

This is one seen JOINT terminal point. It cannot locate original step25–49 conditional-clean,
CFG/native or trajectory causality, or establish blind decoding/FPR/scientific PASS.
CPU and notebook-stub checks do not establish real-model memory or latency.

In [ ]:
from pathlib import Path
import datetime, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'  # Editable branch/tag/ref.
FIXED_CONFIG = {'schema': 'saved-joint-terminal-bridge-v1', 'source_run': '20261009T132316545817Z', 'source_arm': 'JOINT', 'model': {'id': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers', 'revision': '0fad780a534b6463e45facd96134c9f345acfa5b'}, 'carrier': {'key': 'local-joint-state-payload-v1-first-mechanism', 'message_hex': '8001a55a', 'rho': 0.5, 'cap': 1.0}, 'device': 'cuda', 'inputs': {'terminal_latent': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1/20261009T132316545817Z/run/joint/terminal_latent.pt', 'float_rgb': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1/20261009T132316545817Z/run/joint/float_rgb.pt'}}
EXPECTED_COST = {"encode":2,"decode":4,"dit":0,"native":0,"codec":0,"backward":0,"vae_load":1}
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Terminal-Bridge')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / STAMP
OUTPUT.mkdir(exist_ok=False)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-Local-Joint-Terminal-Bridge-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def original_record_failure(stage, exc):
    row = dict(stage=stage, reason=f'{type(exc).__name__}: {exc}', traceback=''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)), retry=False, output=str(OUTPUT), expected_cost=EXPECTED_COST)
    write_json(OUTPUT / 'notebook_failure.json', row)

def logged(command, stage, *, cwd=None, env=None, check=True):
    child = None
    returncode = None
    primary = None
    primary_tb = None
    cleanup_errors = []

    def cleanup_error(label, exc):
        cleanup_errors.append(label + ': ' + repr(exc))
    try:
        with (OUTPUT / 'execution.log').open('a', encoding='utf-8') as log:
            line = 'STAGE ' + stage + ' COMMAND ' + repr(command) + '\n'
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            group = {'start_new_session': True} if os.name == 'posix' else {}
            child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **group)
            try:
                for line in child.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = child.wait()
            except BaseException as exc:
                primary = exc
                primary_tb = exc.__traceback__
            finally:
                if child is not None:
                    if os.name == 'posix':
                        try:
                            os.killpg(child.pid, signal.SIGTERM)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGTERM', exc)
                        deadline = time.monotonic() + 5.0
                        while time.monotonic() < deadline:
                            try:
                                os.killpg(child.pid, 0)
                            except ProcessLookupError:
                                break
                            except BaseException as exc:
                                cleanup_error('group probe', exc)
                                break
                            time.sleep(0.05)
                        try:
                            os.killpg(child.pid, signal.SIGKILL)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGKILL', exc)
                    elif child.poll() is None:
                        try:
                            child.terminate()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('terminate', exc)
                    try:
                        returncode = child.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:
                            child.kill()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('kill', exc)
                        try:
                            returncode = child.wait(timeout=10)
                        except BaseException as exc:
                            cleanup_error('wait after kill', exc)
                    except BaseException as exc:
                        cleanup_error('wait', exc)
                    if child.stdout is not None:
                        try:
                            child.stdout.close()
                        except BaseException as exc:
                            cleanup_error('stdout close', exc)
            if primary is None and (not cleanup_errors):
                log.write('EXIT ' + str(returncode) + '\n')
                log.flush()
    except BaseException as exc:
        if primary is None:
            primary = exc
            primary_tb = exc.__traceback__
        else:
            cleanup_error('log wrapper', exc)
    if primary is None and cleanup_errors:
        primary = RuntimeError('process cleanup failed: ' + '; '.join(cleanup_errors))
        primary_tb = primary.__traceback__
    if primary is None and check and returncode:
        primary = subprocess.CalledProcessError(returncode, command)
        primary_tb = primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary, 'add_note'):
            primary.add_note('secondary cleanup errors: ' + '; '.join(cleanup_errors))
        try:
            record_failure(stage, primary)
        except BaseException as record_error:
            if hasattr(primary, 'add_note'):
                primary.add_note('failure record error: ' + repr(record_error))
        raise primary.with_traceback(primary_tb)
    return returncode

def record_failure(stage, exc):
    original_record_failure(stage, exc)
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
    if (RUN_OUTPUT / 'result.json').exists():
        from runtime.wan.local_joint_terminal_bridge_v1 import finalize_interrupted
        finalize_interrupted(RUN_OUTPUT, 'notebook takeover after child cleanup: ' + repr(exc))

fixed_slots = dict(status='SETUP_STARTED', expected_cost=EXPECTED_COST,
    stages={name: dict(status='NOT_RUN', expected_windows_per_view=88, expected_chips_per_view=1408)
            for name in ('base','candidate','posterior_reconstruction','raw_reinjection','masked_reinjection','capped_reinjection')})
write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
write_json(OUTPUT / 'setup_receipt.json', dict(config=FIXED_CONFIG, expected_cost=EXPECTED_COST,
    output=str(OUTPUT), python=sys.version, scientific_pass=False, automatic_retry=False))
print('Saved-terminal diagnostic output:', OUTPUT, flush=True)

In [ ]:
import io, urllib.request, zipfile
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_state_payload_v1_terminal_bridge_portable_source.zip'
try:
    with urllib.request.urlopen(SOURCE_URL) as response:
        package_bytes = response.read()
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise RuntimeError('unsafe source archive path')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    sys.path.insert(0, str(WORKSPACE))
    print('Source:', SOURCE_URL, 'Workspace:', WORKSPACE, flush=True)
    try:
        write_json(OUTPUT / 'source_receipt.json', dict(url=SOURCE_URL, workspace=str(WORKSPACE)))
    except Exception as exc:
        print('Source note unavailable:', repr(exc), flush=True)
except BaseException as exc:
    record_failure('SOURCE_DOWNLOAD', exc)
    raise

In [ ]:
import importlib.metadata, shutil
try:
    probe_path = OUTPUT / 'dependency_probe.json'
    probe_lines = ['import importlib.metadata as metadata, json, pathlib, shutil, sys, torch, diffusers', 'from diffusers import AutoencoderKLWan', "pins = {'torch':'2.11.0','diffusers':'0.39.0','transformers':'4.57.6','numpy':'2.1.3','accelerate':'1.15.0','safetensors':'0.8.0','huggingface-hub':'0.36.2','tokenizers':'0.22.2','sentencepiece':'0.2.2','ftfy':'6.3.1'}", 'actual = {}', 'for name in pins:', '    try: actual[name] = metadata.version(name)', '    except metadata.PackageNotFoundError: actual[name] = None', "version_differences = {name: {'reference': version, 'actual': actual[name]} for name, version in pins.items() if str(actual[name]).split('+',1)[0] != version}", "assert torch.cuda.is_available(), 'CUDA torch required'", "row = dict(python=sys.version, versions=actual, version_differences=version_differences, torch_cuda_runtime=torch.version.cuda, cuda_available=True, device=torch.cuda.get_device_name(0), total_device_memory=int(torch.cuda.get_device_properties(0).total_memory), ffmpeg=shutil.which('ffmpeg'), ffprobe=shutil.which('ffprobe'))", 'pathlib.Path(' + repr(str(probe_path)) + ").write_text(json.dumps(row, indent=2) + chr(10), encoding='utf-8')"]
    probe_code = '\n'.join(probe_lines)
    probe_returncode = logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_PROBE', check=False)
    if probe_returncode:
        logged([PYTHON, '-m', 'pip', 'install', 'torch==2.11.0', 'diffusers==0.39.0', 'transformers==4.57.6', 'numpy==2.1.3', 'accelerate==1.15.0', 'safetensors==0.8.0', 'huggingface-hub==0.36.2', 'tokenizers==0.22.2', 'sentencepiece==0.2.2', 'ftfy==6.3.1'], 'DEPENDENCY_REPAIR')
        logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_REPROBE')
    dependency_check_returncode = logged([PYTHON, '-m', 'pip', 'check'], 'DEPENDENCY_REPORT', check=False)
    (OUTPUT / 'environment_freeze.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
    environment_receipt = json.loads(probe_path.read_text(encoding='utf-8'))
    environment_receipt.update(dependency_check_returncode=dependency_check_returncode, gpu_name_is_not_a_gate=True, current_joint_compatibility_reference='ac111d0 runtime: torch2.11.0+cu130 diffusers0.39.0', historical_residency_reference='N2 multistep: Python3.13.15 torch2.11.0+cu128 diffusers0.40.0 L4', resource_claim='saved-terminal 2E/4D/one-VAE-load memory and latency remain unverified')
    write_json(OUTPUT / 'environment_receipt.json', environment_receipt)
except BaseException as exc:
    try:
        record_failure('ENVIRONMENT_SETUP', exc)
    except BaseException as record_error:
        if hasattr(exc, 'add_note'):
            exc.add_note('failure record error: ' + repr(record_error))
    raise

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
except Exception as exc:
    print('Optional HF token unavailable:', type(exc).__name__, flush=True)

run_returncode = None
try:
    command = [PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_terminal_bridge_v1_run',
               '--config', str(WORKSPACE / 'experiments/wan_state_clock/configs/local_joint_terminal_bridge_v1.json'), '--output', str(RUN_OUTPUT)]
    run_returncode = logged(command, 'FIXED_TERMINAL_BRIDGE', cwd=WORKSPACE, check=False)
    from runtime.wan.local_joint_terminal_bridge_v1 import finalize_interrupted
    # The logger has reaped the child/process group before any takeover writes.
    finalize_interrupted(RUN_OUTPUT, 'child exited with return code ' + str(run_returncode))
    result_path = RUN_OUTPUT / 'result.json'
    result = json.loads(result_path.read_text(encoding='utf-8')) if result_path.exists() else None
    write_json(OUTPUT / 'execution_receipt.json', dict(returncode=run_returncode,
        status=None if result is None else result['status'],
        actual_counts=None if result is None else result['counts'], expected_cost=EXPECTED_COST,
        scientific_pass=False, automatic_retry=False))
    if run_returncode != 0 or result is None or result['status'] != 'COMPLETE':
        raise RuntimeError('fixed terminal bridge incomplete; retain entire output: ' + str(OUTPUT))
    fixed_slots.update(status='SUPERSEDED_BY_RUN_RESULT', result='run/result.json')
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
except BaseException as exc:
    try: record_failure('FIXED_TERMINAL_BRIDGE', exc)
    except BaseException as record_error:
        if hasattr(exc, 'add_note'): exc.add_note('failure record error: ' + repr(record_error))
    raise

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text(encoding='utf-8'))
print('Engineering status:', result['status'])
print('Actual calls:', result['counts'])
for name, stage in result['stages'].items():
    row = stage['views']['postclip']
    means = row.get('payload_signed_means', [])
    print(name, stage['status'], 'state gap=', row.get('state_gap'),
          'positive payload=', sum(x is not None and x > 0 for x in means), '/32',
          'missing payload=', 32-sum(x is not None for x in means))
print(result['ceiling'])
print('Return the entire output directory, including failures:', OUTPUT)